# PRODUCTION

In [ ]:
from pathlib import Path
import pandas as pd

def read_parquet(path_to_file, display_command):
    # Set your parquet file path here (works from project root or pdf_reader/)
    parquet_rel = Path(path_to_file)
    parquet_path = parquet_rel if parquet_rel.exists() else Path("..") / parquet_rel
    parquet_path = parquet_path.resolve()

    # Read parquet into a DataFrame ("unpack")
    df = pd.read_parquet(parquet_path)

    # Quick check
    print(f"Using: {parquet_path}")
    print("Unpackaging compleet.")
    
    commands = ["1", "yes", "Yes", "YES", "y", "Y"]
    print()
    print("To display input: 1, yes, Yes, YES, y, Y")
    user = input()

    #if display_command in commands:
    if user in commands:
        print(f"data shape: {df.shape}")
        display(df.head())
    else:
        print(f"data shape: {df.shape}")

    # Optional: export to CSV if needed
    # df.to_csv(parquet_path.with_suffix(".csv"), index=False)

read_parquet("Data/Exploring_meta/thesis_meta_combined.parquet", 0)



Using: /Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/Data/Exploring_meta/thesis_meta_combined.parquet
Unpackaging compleet.

To display input: 1, yes, Yes, YES, y, Y


data shape: (19690, 40)


# EXPLORATION

In [60]:
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 19690 entries, 0 to 19689
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   abstract_ts                   1073 non-null   str    
 1   access_ss                     19690 non-null  str    
 2   Affiliations                  2876 non-null   str    
 3   Timestamp                     19690 non-null  str    
 4   Author                        19682 non-null  str    
 5   citation_count_i              0 non-null      float64
 6   ID                            19690 non-null  int64  
 7   dtu_library_collection_facet  0 non-null      float64
 8   collection_facet              17783 non-null  str    
 9   Publication Year              19690 non-null  int64  
 10  Conference                    0 non-null      float64
 11  DOI                           0 non-null      float64
 12  Editor                        0 non-null      float64
 13  embargo_ssf 

None

# DEVELOPMENT

In [61]:
df[df['Publisher'].notna()]['Publisher']
df[df['Affiliations'].notna()]['Affiliations']

0        Depatment of Systems Biology, Technical Univer...
1        [affiliation name could not be established]|Au...
2        [affiliation name could not be established]|In...
3              Ørsted•DTU, Technical University of Denmark
4        Intelligent Signal Processing, Informatics and...
                               ...                        
19678    [affiliation name could not be established]|In...
19679    [affiliation name could not be established]|Op...
19680    [affiliation name could not be established]|Tr...
19681    [affiliation name could not be established]|La...
19682    [affiliation name could not be established]|Op...
Name: Affiliations, Length: 2876, dtype: str

In [62]:
unique_affiliations = (
    df['Affiliations']
    .dropna()
    .str.replace('[affiliation name could not be established]|', '', regex=False)
    .str.replace('|[affiliation unknown]', '', regex=False)
    .str.strip('|')
    .unique()
)
print("Unique Affiliations:", len(unique_affiliations))
print(unique_affiliations)

Unique Affiliations: 462
<ArrowStringArray>
[                                                                                                                                                      'Depatment of Systems Biology, Technical University of Denmark|Novozymes A/S|Department of Systems Biology, Technical University of Denmark',
                                                                                                                                                                                                                                          'Automation, Ørsted DTU, Technical University of Denmark',
                                                                                                                                                                                                                   'Institut for Planlægning, Innovation og Ledelse, Danmarks Tekniske Universitet',
                                                                             

In [63]:
import re

pattern = (
    r"(Department of [^|]*?, Technical University of Denmark|"
    r"Institut for [^|]*?, Danmarks Tekniske Universitet)"
)

cleaned_affiliations = (
    pd.Series(unique_affiliations)
    .str.findall(pattern)
    .explode()
    .dropna()
    .drop_duplicates()
    #.str.replace(', Technical University of Denmark', '', regex=False)
    #.str.replace(', Danmarks Tekniske Universitet', '', regex=False)
    .to_numpy()
)

print("Cleaned Affiliations:", len(cleaned_affiliations))
print(cleaned_affiliations)

Cleaned Affiliations: 35
['Department of Systems Biology, Technical University of Denmark'
 'Institut for Planlægning, Innovation og Ledelse, Danmarks Tekniske Universitet'
 'Department of Transport, Technical University of Denmark'
 'Department of Informatics and Mathematical Modeling, Technical University of Denmark'
 'Department of Chemical Engineering, Technical University of Denmark'
 'Department of Chemical and Biochemical Engineering, Technical University of Denmark'
 'Department of Electrical Engineering, Technical University of Denmark'
 'Institut for Kommunikation, Optik & Materialer, Danmarks Tekniske Universitet'
 'Department of Micro and Nanotechnology, Technical University of Denmark'
 'Department of Manufacturing Engineering and Management, Technical University of Denmark'
 'Institut for Informatik og Matematisk Modellering, Danmarks Tekniske Universitet'
 'Department of Communications, Optics & Materials, Technical University of Denmark'
 'Department of Mechanical Engin

In [27]:
unique_publishers = df['Publisher'].dropna().unique()
print("Unique Publishers:", len(unique_publishers))
print(unique_publishers)

Unique Publishers: 67
<ArrowStringArray>
[                                                       'Department of Informatics and Mathematical Modeling, Technical University of Denmark, DTU',
                                                                   'Danmarks Tekniske Universitet, Risø Nationallaboratoriet for Bæredygtig Energi',
                                                                                                            'Technical University of Denmark (DTU)',
                                                                                                                                              'IPL',
                                                            'Institut for Informatik og Matematisk Modellering, Danmarks Tekniske Universitet, DTU',
                                                                                               'DTU Department of Civil and Mechanical Engineering',
                                                                 